# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

Finding A: ML Appendix — Growth & Classification (logistic regression, 71% holdout accuracy)

The paper reports a logistic regression predicting growth vs. decline, with content_age, days_since_update, and days_visible as the top coefficients, validated with an 80/20 holdout split. My methodology question: the paper states the dataset spans 57 brands is the 80/20 split a random row-level split, or a brand-grouped split? If it's a random row split, pages from the same brand could appear in both train and test, letting the model partially learn brand-specific quirks rather than a pattern that generalizes to a brand it hasn't seen. This mirrors the exact leakage lesson from my own ML-04/ML-08 work I'd want to know whether "71% holdout accuracy" would hold up under a client/brand-holdout split, the same way my own Random Forest result (0.72 Precision@50) only became trustworthy once I re-tested it under a client-grouped split rather than a naive one.

Finding B: ML Appendix — Feature Importance for Health Score (Random Forest)

The paper is admirably upfront that "the target itself is partly constructed from some of these inputs" Average Position (43%) and Impressions (32%) dominate importance, and Health Score's own formula is literally built from position, impressions, CTR, and scroll depth. My methodology question: given that overlap, is this really measuring "what predicts health" or mostly re-deriving the known formula weights the model was trained to reconstruct? The paper does flag this ("descriptive rather than causal"), which I think is the right caution but I'd ask whether reporting a feature-importance chart at all risks readers skimming past that caveat and treating position/impressions as newly "discovered" levers, when they were definitionally baked into the target from the start. A cleaner version might drop Position/Impressions/CTR/Scroll from the feature set entirely and see what, if anything, still predicts Health from truly independent signals.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [3]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/sarmad341/flyrank-internship-01"
REPO_DIR = "flyrank-internship-01"
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

import pandas as pd, numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit, train_test_split

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
features = ["impressions_90d", "days_since_last_update", "avg_position", "ctr", "word_count", "content_age_days"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df["is_declining_label"]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# BEFORE: naive random split (ignores client grouping)
Xtr_r, Xte_r, ytr_r, yte_r = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
rf_random = RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced").fit(Xtr_r, ytr_r)
p50_random = precision_at_k(rf_random.predict_proba(Xte_r)[:,1], yte_r.values, 50)

# AFTER: honest client-holdout split (same as ML-08)
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df["client_id"]))
Xtr_g, Xte_g = X.iloc[train_idx], X.iloc[test_idx]
ytr_g, yte_g = y.iloc[train_idx], y.iloc[test_idx]
rf_grouped = RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced").fit(Xtr_g, ytr_g)
p50_grouped = precision_at_k(rf_grouped.predict_proba(Xte_g)[:,1], yte_g.values, 50)

print(f"BEFORE (naive random split)   Precision@50: {p50_random:.3f}")
print(f"AFTER  (client-holdout split) Precision@50: {p50_grouped:.3f}")

BEFORE (naive random split)   Precision@50: 0.900
AFTER  (client-holdout split) Precision@50: 0.720


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [4]:
# Confirm none of these excluded columns snuck into the feature set
excluded = ["trend_direction", "trend_pct", "is_declining_label"]
print("Any excluded/label columns in features?", any(c in features for c in excluded))
print("Features used:", features)

# Check for near-duplicate/derived relationships (e.g. accidental correlation with label)
for f in features:
    corr = df[f].corr(df["is_declining_label"])
    print(f"{f}: correlation with label = {corr:.3f}")

Any excluded/label columns in features? False
Features used: ['impressions_90d', 'days_since_last_update', 'avg_position', 'ctr', 'word_count', 'content_age_days']
impressions_90d: correlation with label = -0.018
days_since_last_update: correlation with label = 0.081
avg_position: correlation with label = -0.029
ctr: correlation with label = -0.062
word_count: correlation with label = 0.090
content_age_days: correlation with label = -0.164


None of my six features (impressions_90d, days_since_last_update, avg_position, ctr, word_count, content_age_days) include trend_direction or trend_pct, so the ML-04 leakage lesson holds here — no direct label-derived column. Correlations above are all modest, consistent with genuine signal rather than a hidden shortcut.

## 4. Claim rewrite

Original bold claim: "Random Forest beats the baseline." Rewritten in safe language: On this starter data slice, under a client-holdout validation split, Random Forest showed a directional improvement over the Week-4 baseline rule (Precision@50 of 0.72 vs. 0.62) this is an observed, decision-support result on a 30,000-row teaching slice, not a proven causal claim, and it has not yet been validated against the full 79M-row warehouse.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.